# Emberline Sri Lanka sensor-transfer research

Research only. Uses the verified observed-data ZIP; no NASA downloads or login. It does not fill gaps, certify observing coverage, reconstruct early bands or modify the frontend. Run all cells, then download one results ZIP. This reproduces the regional vegetation-only experiment; it does not validate district allocation or other category scopes.

In [88]:
%pip -q install numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 pyarrow==26.0.1 h3==4.5.0


## Load the verified input
Set `INPUT_ZIP` to your file location in VS Code. In Colab, upload the supplied 14.8 MB `emberline_atlas_lk_observed_2000_2024_v1.zip` when prompted. Do not use the original historical download ZIP. The exact ZIP and every inner file are verified before use.

In [89]:
from pathlib import Path
import hashlib,json,zipfile,sys
INPUT_ZIP=Path('emberline_atlas_lk_observed_2000_2024_v1.zip')
if not INPUT_ZIP.exists():
    project_copy=Path('data/atlas')/INPUT_ZIP.name
    if project_copy.exists():INPUT_ZIP=project_copy
    elif 'google.colab' in sys.modules:
        from google.colab import files
        uploaded=files.upload()
        assert INPUT_ZIP.name in uploaded,'Upload the verified observed-data input ZIP.'
    else:raise FileNotFoundError('Set INPUT_ZIP to your observed-data ZIP location and rerun.')
assert hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()=='90cb8ffb2c3c86559fea5453834bc6197a4972ea89630a697135e7a8d244a291','Wrong or changed input ZIP'
WORK=Path('emberline_transfer_research_workspace');WORK.mkdir(exist_ok=True)
DATA=WORK/'verified_inputs';DATA.mkdir(exist_ok=True)
with zipfile.ZipFile(INPUT_ZIP) as archive:
    names=archive.namelist()
    assert len(names)==len(set(names)),'Duplicate archive entries'
    bundle=json.loads(archive.read('analysis_bundle_manifest.json'))
    assert set(names)==set(bundle['files'])|{'analysis_bundle_manifest.json'}
    for name,receipt in bundle['files'].items():
        assert Path(name).name==name and name not in ('.','..'),'Unsafe member name'
        payload=archive.read(name)
        assert len(payload)==receipt['bytes'] and hashlib.sha256(payload).hexdigest()==receipt['sha256'],name
        (DATA/name).write_bytes(payload)
(DATA/'manifest.json').write_bytes((DATA/'dataset_manifest.json').read_bytes())
print('Verified:',bundle['observations'],'standard observations. Coverage remains unverified.')


KeyboardInterrupt: 

## Frozen method
Primary: vegetation, all native confidence levels, H3 resolution 8, conservative sourced gap-date exclusions. Seven configurations are prespecified. Whole-year tests use separate fit, selection and calibration periods. The chronological test may withhold unsupported years/tasks; inspect actual tested months. Calendar alignment does not establish matched observing opportunities. Empirical interval coverage has no guaranteed nominal level under temporal dependence. The next cell embeds the same reviewed research implementation used locally.

In [ ]:
research_namespace={'__file__':str((WORK/'pipeline'/'research.py').resolve()),'__name__':'emberline_research'}
exec(compile('"""Regional observed-activity transfer research on the repaired dataset.\nCalendar support is not matched overpasses or cloud/observation completeness.\nNo production estimates, local allocation, satellite replacement or season claims.\n"""\nfrom pathlib import Path\nimport hashlib,json,math\nimport numpy as np\nimport pandas as pd\nimport h3\nfrom scipy.optimize import nnls\nfrom scipy.stats import spearmanr\n\nROOT=Path(__file__).resolve().parent.parent\nDATA=ROOT/\'data/atlas/canonical_lk_2000_2024_v1\'\nOUT=ROOT/\'data/atlas/transfer_research_v1\'\nTASK_SENSORS={\'terra_to_modis\':[\'Terra\',\'Aqua\'],\'modis_to_snpp\':[\'Terra\',\'Aqua\',\'Suomi-NPP\'],\n \'terra_chain_to_snpp\':[\'Terra\',\'Aqua\',\'Suomi-NPP\'],\'noaa20_to_snpp_diagnostic\':[\'NOAA-20\',\'Suomi-NPP\']}\nCONFIGS=[{\'name\':\'primary\',\'resolution\':8,\'filter\':\'vegetation_all_confidence\',\'gap_mask\':\'all_documented_global_gap_dates\',\'window\':\'full\'},\n {\'name\':\'coarser_grid\',\'resolution\':7,\'filter\':\'vegetation_all_confidence\',\'gap_mask\':\'all_documented_global_gap_dates\',\'window\':\'full\'},\n {\'name\':\'finer_grid\',\'resolution\':9,\'filter\':\'vegetation_all_confidence\',\'gap_mask\':\'all_documented_global_gap_dates\',\'window\':\'full\'},\n {\'name\':\'native_confidence_screen\',\'resolution\':8,\'filter\':\'vegetation_native_screen\',\'gap_mask\':\'all_documented_global_gap_dates\',\'window\':\'full\'},\n {\'name\':\'listed_missing_dates_only\',\'resolution\':8,\'filter\':\'vegetation_all_confidence\',\'gap_mask\':\'listed_missing_dates_only\',\'window\':\'full\'},\n {\'name\':\'last5_fit_years\',\'resolution\':8,\'filter\':\'vegetation_all_confidence\',\'gap_mask\':\'all_documented_global_gap_dates\',\'window\':\'last5\'},\n {\'name\':\'last8_fit_years\',\'resolution\':8,\'filter\':\'vegetation_all_confidence\',\'gap_mask\':\'all_documented_global_gap_dates\',\'window\':\'last8\'}]\nPOLICY={\'research_id\':\'lk_observed_transfer_research_v1\',\'scope\':\'Sri Lanka original study polygon; regional only\',\n \'target\':\'reference-sensor observed vegetation active-cell-days on common eligible UTC calendar dates\',\n \'primary_configuration\':CONFIGS[0],\'sensitivity_design\':\'one factor at a time; no resolution/filter chosen from outer test scores\',\n \'minimum_supported_days_per_month\':20,\'minimum_supported_calendar_fraction\':0.8,\'minimum_months_per_year\':8,\n \'meaning_of_support\':\'verified archive queries with selected global gap-date exclusions; not verified observing opportunities\',\n \'native_screen\':\'MODIS confidence_pct >=30; VIIRS n/h; filtering conventions, not equal sensor probabilities\',\n \'monthly_interval_method\':\'separate two-year calibration block; empirical scaled absolute-residual quantiles; nominal guarantees not asserted under temporal dependence\',\n \'union_constraint\':\'combined MODIS predictions are floored at contemporaneous observed Terra activity; the Terra-target seasonal baseline has the same floor\',\n \'baselines\':[\'seasonal_daily_climatology (Terra floor when target is union MODIS)\',\'uncalibrated_native_source_identity\'],\n \'year_split\':\'outer held-out whole years; model selection confined to fit years; separate calibration years never used for selection or refitting\',\n \'chronological_test\':\'fit/selection/calibration use years <=2021; test 2022–2024 contemporaneous sensor transfer, not forecasting\',\n \'annual_intervals\':\'withheld: two calibration years are insufficient for supported 80/90 percent annual uncertainty\',\n \'noaa20_status\':\'calendar-aligned diagnostic only; matched observing footprints/overpasses unverified, no gap filling enabled\',\n \'coverage_verified\':False,\'physical_fire_truth_validated\':False,\'production_export_enabled\':False,\'frontend_changed\':False}\nCANDIDATES=[\'global_ratio\',\'seasonal_ratio\',\'positive_affine\']\n\ndef digest(raw):return hashlib.sha256(raw).hexdigest()\n\ndef load_inputs(path=None):\n    path=Path(DATA if path is None else path);m=json.loads((path/\'manifest.json\').read_text())\n    if m[\'observations\']!=171244 or m[\'production_export_enabled\'] is not False:raise ValueError(\'Unexpected observed-data checkpoint\')\n    for name,r in m[\'files\'].items():\n        raw=(path/name).read_bytes()\n        if len(raw)!=r[\'bytes\'] or digest(raw)!=r[\'sha256\']:raise ValueError(\'Observed checkpoint file changed\')\n    p=pd.read_parquet(path/\'observations.parquet\');daily=pd.read_parquet(path/\'daily_status.parquet\')\n    if not p.quality.eq(\'sp\').all() or not daily.observing_coverage.eq(\'unverified\').all():raise ValueError(\'Wrong source quality or unsupported coverage claim\')\n    return p,daily,m\n\ndef activity_sets(points,resolution,filter_name):\n    p=points[points.type.eq(0)&points.satellite.isin([\'Terra\',\'Aqua\',\'Suomi-NPP\',\'NOAA-20\'])].copy()\n    if filter_name==\'vegetation_native_screen\':\n        modis=p.satellite.isin([\'Terra\',\'Aqua\']);p=p[(modis&p.confidence_pct.ge(30))|(~modis&p.confidence.isin([\'n\',\'h\']))]\n    elif filter_name!=\'vegetation_all_confidence\':raise ValueError(\'Unknown native filter\')\n    p[\'cell\']=[h3.latlng_to_cell(float(lat),float(lon),resolution) for lat,lon in zip(p.latitude,p.longitude)]\n    # One observation of activity per sensor/cell/UTC day; no deduplication across days.\n    unique=p[[\'acq_date\',\'satellite\',\'cell\']].drop_duplicates()\n    sets={}\n    for (day,satellite),f in unique.groupby([\'acq_date\',\'satellite\'],sort=False):sets[(day,satellite)]=set(f.cell)\n    return sets,{\'filtered_pixel_observations\':len(p),\'sensor_cell_days\':len(unique),\'resolution\':resolution,\'filter\':filter_name}\n\ndef qualify_months(daily,sets,gap_mask):\n    d=daily[daily.satellite.isin([\'Terra\',\'Aqua\',\'Suomi-NPP\',\'NOAA-20\'])].copy()\n    valid=d.recorded_detections.notna()&~d.upstream_lists_missing_standard_data\n    if gap_mask==\'all_documented_global_gap_dates\':valid&=~d.documented_global_gap_on_date\n    elif gap_mask!=\'listed_missing_dates_only\':raise ValueError(\'Unknown gap mask\')\n    d[\'eligible_calendar_date\']=valid\n    states=d.pivot(index=\'date\',columns=\'satellite\',values=\'eligible_calendar_date\').fillna(False)\n    records=[]\n    for task,sensors in TASK_SENSORS.items():\n        for month in sorted({str(day)[:7] for day in states.index}):\n            dates=[day for day in states.index if str(day).startswith(month)]\n            eligible=[day for day in dates if all(bool(states.loc[day,s]) for s in sensors)]\n            days_in_month=pd.Period(month,freq=\'M\').days_in_month\n            good=len(eligible)>=20 and len(eligible)/days_in_month>=.8\n            t=a=combined=s=n20=duplicates=0\n            for day in eligible:\n                ts=sets.get((day,\'Terra\'),set());aa=sets.get((day,\'Aqua\'),set())\n                t+=len(ts);a+=len(aa);combined+=len(ts|aa);duplicates+=len(ts&aa)\n                s+=len(sets.get((day,\'Suomi-NPP\'),set()));n20+=len(sets.get((day,\'NOAA-20\'),set()))\n            records.append({\'task\':task,\'month\':month,\'year\':int(month[:4]),\'month_number\':int(month[5:]),\n                \'days_in_month\':days_in_month,\'common_calendar_days\':len(eligible),\'calendar_fraction\':len(eligible)/days_in_month,\n                \'eligible_month\':good,\'eligibility_reason\':\'calendar_support_pass\' if good else \'insufficient_or_unsupported_common_calendar_dates\',\n                \'terra\':t if eligible else np.nan,\'aqua\':a if eligible else np.nan,\'modis\':combined if eligible else np.nan,\n                \'snpp\':s if eligible else np.nan,\'noaa20\':n20 if eligible else np.nan,\'terra_aqua_overlap_cell_days\':duplicates if eligible else np.nan,\n                \'observing_opportunities_verified\':False,\'gap_mask\':gap_mask})\n    frame=pd.DataFrame(records)\n    frame[\'eligible_year\']=False\n    # First incomplete reference/launch years remain outside primary annual-cohort definitions.\n    for task in TASK_SENSORS:\n        first=2003 if task==\'terra_to_modis\' else 2013 if task!=\'noaa20_to_snpp_diagnostic\' else 2018\n        counts=frame[frame.task.eq(task)&frame.eligible_month&frame.year.ge(first)].groupby(\'year\').size()\n        frame.loc[frame.task.eq(task)&frame.year.isin(counts[counts.ge(8)].index),\'eligible_year\']=True\n    return frame\n\ndef fit_model(frame,xcol,ycol,kind):\n    x=frame[xcol].to_numpy(float);y=frame[ycol].to_numpy(float);days=frame.common_calendar_days.to_numpy(float)\n    if len(frame)<12 or not np.isfinite(x).all() or not np.isfinite(y).all() or y.sum()<=0:return None\n    if kind==\'seasonal_baseline\':\n        rate=y.sum()/days.sum();rates={m:(frame.loc[frame.month_number.eq(m),ycol].sum()+90*rate)/(frame.loc[frame.month_number.eq(m),\'common_calendar_days\'].sum()+90) for m in range(1,13)}\n        return {\'kind\':kind,\'rates\':rates}\n    if kind in [\'global_ratio\',\'seasonal_ratio\']:\n        if x.sum()<=0:return None\n        ratio=y.sum()/x.sum();model={\'kind\':kind,\'ratio\':ratio}\n        if kind==\'seasonal_ratio\':\n            prior=max(1.,float(np.median(x[x>0]))*3)\n            model[\'rates\']={m:(frame.loc[frame.month_number.eq(m),ycol].sum()+prior*ratio)/(frame.loc[frame.month_number.eq(m),xcol].sum()+prior) for m in range(1,13)}\n        return model\n    if kind==\'positive_affine\':\n        design=np.column_stack([np.ones(len(x)),x/days]);weights=np.sqrt(days)\n        coefficients,_=nnls(design*weights[:,None],y/days*weights)\n        return {\'kind\':kind,\'intercept\':float(coefficients[0]),\'slope\':float(coefficients[1])}\n    raise ValueError(\'Unknown transfer candidate\')\n\ndef predict_model(model,frame,xcol):\n    if model is None:return None\n    x=frame[xcol].to_numpy(float)\n    if model[\'kind\']==\'seasonal_baseline\':return np.array([model[\'rates\'][int(m)] for m in frame.month_number])*frame.common_calendar_days.to_numpy(float)\n    if model[\'kind\']==\'global_ratio\':return x*model[\'ratio\']\n    if model[\'kind\']==\'seasonal_ratio\':return x*np.array([model[\'rates\'][int(m)] for m in frame.month_number])\n    return model[\'intercept\']*frame.common_calendar_days.to_numpy(float)+model[\'slope\']*x\n\ndef task_columns(task):return (\'terra\',\'modis\') if task==\'terra_to_modis\' else (\'noaa20\',\'snpp\') if task==\'noaa20_to_snpp_diagnostic\' else (\'modis\',\'snpp\')\n\ndef fit_candidate(frame,task,candidate):\n    if task==\'terra_chain_to_snpp\':\n        first,second=candidate.split(\'|\');a=fit_model(frame,\'terra\',\'modis\',first);b=fit_model(frame,\'modis\',\'snpp\',second)\n        return None if a is None or b is None else (a,b)\n    x,y=task_columns(task);return fit_model(frame,x,y,candidate)\n\ndef predict_candidate(model,frame,task):\n    if model is None:return None\n    if task==\'terra_chain_to_snpp\':\n        intermediate=np.maximum(predict_model(model[0],frame,\'terra\'),frame.terra.to_numpy(float));f=frame.copy();f[\'estimated_modis\']=intermediate\n        return predict_model(model[1],f,\'estimated_modis\')\n    x,_=task_columns(task);prediction=predict_model(model,frame,x)\n    return np.maximum(prediction,frame.terra.to_numpy(float)) if task==\'terra_to_modis\' else prediction\n\ndef select_candidate(fit,task):\n    years=sorted(fit.year.unique());candidates=[f\'{a}|{b}\' for a in CANDIDATES for b in CANDIDATES] if task==\'terra_chain_to_snpp\' else CANDIDATES\n    target=\'snpp\' if task==\'terra_chain_to_snpp\' else task_columns(task)[1]\n    scores=[]\n    for order,candidate in enumerate(candidates):\n        error=weight=0.;folds=0\n        for year in years:\n            train=fit[fit.year.ne(year)];test=fit[fit.year.eq(year)]\n            model=fit_candidate(train,task,candidate);prediction=predict_candidate(model,test,task)\n            if prediction is None:continue\n            error+=float(np.abs(prediction-test[target].to_numpy(float)).sum());weight+=float(test[target].sum());folds+=1\n        score=error/weight if weight>0 and folds==len(years) else np.inf\n        scores.append((score,order,candidate))\n    best=min(scores)\n    return None if not np.isfinite(best[0]) else {\'candidate\':best[2],\'inner_wape\':best[0],\'inner_years\':years,\'candidate_scores\':{c:float(s) if np.isfinite(s) else None for s,_,c in scores}}\n\ndef empirical_radius(observed,prediction,level):\n    scores=np.abs(np.asarray(observed,float)-np.asarray(prediction,float))/np.sqrt(1+np.asarray(prediction,float))\n    n=len(scores);rank=math.ceil((n+1)*level)\n    if n<18 or rank>n:return None\n    return float(np.sort(scores)[rank-1])\n\ndef evaluate_split(frame,task,test_years,window=\'full\',chronological=False):\n    test=frame[frame.year.isin(test_years)].copy();train=frame[~frame.year.isin(test_years)]\n    if chronological:train=train[train.year.lt(min(test_years))]\n    train_years=sorted(train.year.unique());calibration_years=train_years[-2:];fit_years=train_years[:-2]\n    if window!=\'full\':fit_years=fit_years[-int(window.removeprefix(\'last\')):]\n    if len(fit_years)<4 or len(test)==0:return [],{\'status\':\'insufficient_fit_years\',\'test_years\':list(map(int,test_years)),\'fit_years\':list(map(int,fit_years))}\n    fit=train[train.year.isin(fit_years)];cal=train[train.year.isin(calibration_years)]\n    if set(fit.year)&set(cal.year) or set(test.year)&(set(fit.year)|set(cal.year)):raise ValueError(\'Year split leakage\')\n    choice=select_candidate(fit,task)\n    if choice is None:return [],{\'status\':\'no_supported_transfer_candidate\',\'test_years\':list(map(int,test_years))}\n    model=fit_candidate(fit,task,choice[\'candidate\']);cal_pred=predict_candidate(model,cal,task);pred=predict_candidate(model,test,task)\n    x,target=task_columns(task)\n    if task==\'terra_chain_to_snpp\':x=\'terra\';target=\'snpp\'\n    baseline=fit_model(fit,x,target,\'seasonal_baseline\');baseline_cal=predict_model(baseline,cal,x);baseline_test=predict_model(baseline,test,x)\n    if task==\'terra_to_modis\':\n        baseline_cal=np.maximum(baseline_cal,cal.terra.to_numpy(float));baseline_test=np.maximum(baseline_test,test.terra.to_numpy(float))\n    records=[]\n    for role,cp,tp in [(\'selected_transfer\',cal_pred,pred),(\'seasonal_baseline\',baseline_cal,baseline_test),(\'native_source_identity\',cal[x].to_numpy(float),test[x].to_numpy(float))]:\n        radii={str(int(level*100)):empirical_radius(cal[target].to_numpy(float),cp,level) for level in [.8,.9]}\n        for pos,row in enumerate(test.itertuples()):\n            point=float(tp[pos]);observed=float(getattr(row,target));r={\'task\':task,\'month\':row.month,\'year\':int(row.year),\'evaluation\':\'chronological_2022_2024\' if chronological else \'outer_heldout_year\',\n              \'role\':role,\'model\':choice[\'candidate\'] if role==\'selected_transfer\' else role,\'observed\':observed,\'prediction\':point,\'residual_observed_minus_prediction\':observed-point,\n              \'common_calendar_days\':int(row.common_calendar_days),\'calendar_fraction\':row.calendar_fraction,\n              \'fit_years_json\':json.dumps(list(map(int,fit_years))),\'calibration_years_json\':json.dumps(list(map(int,calibration_years))),\'test_years_json\':json.dumps(list(map(int,test_years))),\n              \'inner_selection_wape\':choice[\'inner_wape\'] if role==\'selected_transfer\' else np.nan,\'calibration_months\':len(cal),\'interval_status\':\'empirical_separate_calibration; no nominal coverage guarantee\' if all(q is not None for q in radii.values()) else \'withheld_insufficient_calibration_support\'}\n            for level,q in radii.items():\n                half=q*np.sqrt(1+point) if q is not None else np.nan\n                r[\'lower_\'+level]=max(0.,point-half) if q is not None else np.nan;r[\'upper_\'+level]=point+half if q is not None else np.nan\n            records.append(r)\n    audit={\'status\':\'evaluated\',\'task\':task,\'test_years\':list(map(int,test_years)),\'fit_years\':list(map(int,fit_years)),\'calibration_years\':list(map(int,calibration_years)),\n       \'selected_model\':choice[\'candidate\'],\'candidate_inner_scores\':choice[\'candidate_scores\'],\'calibration_months\':len(cal)}\n    return records,audit\n\ndef metrics(frame):\n    y=frame.observed.to_numpy(float);p=frame.prediction.to_numpy(float);den=y.sum()\n    r={\'months\':len(frame),\'test_years\':int(frame.year.nunique()),\'observed_activity\':float(den),\'wape\':float(np.abs(p-y).sum()/den) if den else None,\n      \'relative_bias\':float((p-y).sum()/den) if den else None,\'mae\':float(np.abs(p-y).mean()),\'rmse\':float(np.sqrt(np.mean((p-y)**2)))}\n    for level in [\'80\',\'90\']:\n        valid=frame[\'lower_\'+level].notna()&frame[\'upper_\'+level].notna();r[\'interval_months_\'+level]=int(valid.sum())\n        r[\'empirical_coverage_\'+level]=float(((frame.loc[valid,\'observed\']>=frame.loc[valid,\'lower_\'+level])&(frame.loc[valid,\'observed\']<=frame.loc[valid,\'upper_\'+level])).mean()) if valid.any() else None\n        r[\'mean_width_\'+level]=float((frame.loc[valid,\'upper_\'+level]-frame.loc[valid,\'lower_\'+level]).mean()) if valid.any() else None\n    return r\n\ndef annual_evaluation(predictions):\n    rows=[]\n    for keys,f in predictions.groupby([\'configuration\',\'task\',\'evaluation\',\'role\',\'year\'],sort=False):\n        if f.month.nunique()!=12 or len(f)!=12:continue\n        config,task,evaluation,role,year=keys\n        rows.append({\'configuration\':config,\'task\':task,\'evaluation\':evaluation,\'role\':role,\'year\':int(year),\'observed\':float(f.observed.sum()),\'prediction\':float(f.prediction.sum()),\n         \'common_calendar_days\':int(f.common_calendar_days.sum()),\'annual_interval_status\':\'withheld; insufficient independent calibration years; monthly bounds not summed\'})\n    annual=pd.DataFrame(rows);summaries=[]\n    if len(annual):\n        for keys,f in annual.groupby([\'configuration\',\'task\',\'evaluation\',\'role\'],sort=False):\n            config,task,evaluation,role=keys;den=f.observed.sum();rank=float(spearmanr(f.observed,f.prediction).statistic) if len(f)>=3 and f.observed.nunique()>1 and f.prediction.nunique()>1 else None\n            summaries.append({\'configuration\':config,\'task\':task,\'evaluation\':evaluation,\'role\':role,\'full_12_month_years\':len(f),\n              \'annual_common_support_wape\':float((f.prediction-f.observed).abs().sum()/den) if den else None,\'annual_relative_bias\':float((f.prediction-f.observed).sum()/den) if den else None,\n              \'annual_rank_spearman\':rank,\'annual_interval_status\':\'withheld\',\'scope\':\'sum on common supported calendar dates, not full physical-fire exposure\'})\n    return annual,pd.DataFrame(summaries)\n\ndef run(output=None,summary_path=None):\n    output=Path(OUT if output is None else output);output.mkdir(parents=True,exist_ok=True);p,d,manifest=load_inputs();cache={};predictions=[];audits=[];qualifications=[];activity_stats=[]\n    for config in CONFIGS:\n        key=(config[\'resolution\'],config[\'filter\'])\n        if key not in cache:cache[key]=activity_sets(p,*key)\n        sets,stats=cache[key];q=qualify_months(d,sets,config[\'gap_mask\']);q[\'configuration\']=config[\'name\'];qualifications.append(q);activity_stats.append({\'configuration\':config[\'name\'],**stats})\n        for task in TASK_SENSORS:\n            data=q[q.task.eq(task)&q.eligible_month&q.eligible_year].copy();years=sorted(data.year.unique())\n            for year in years:\n                rows,audit=evaluate_split(data,task,[year],config[\'window\']);predictions.extend([{**r,\'configuration\':config[\'name\'],\'resolution\':config[\'resolution\'],\'native_filter\':config[\'filter\'],\'gap_mask\':config[\'gap_mask\'],\'training_window\':config[\'window\']} for r in rows]);audits.append({\'configuration\':config[\'name\'],\'task\':task,\'evaluation\':\'outer_heldout_year\',**audit})\n            rows,audit=evaluate_split(data,task,[2022,2023,2024],config[\'window\'],True)\n            predictions.extend([{**r,\'configuration\':config[\'name\'],\'resolution\':config[\'resolution\'],\'native_filter\':config[\'filter\'],\'gap_mask\':config[\'gap_mask\'],\'training_window\':config[\'window\']} for r in rows]);audits.append({\'configuration\':config[\'name\'],\'task\':task,\'evaluation\':\'chronological_2022_2024\',**audit})\n        print(\'Evaluated configuration:\',config[\'name\'],flush=True)\n    predictions=pd.DataFrame(predictions);qualification=pd.concat(qualifications,ignore_index=True)\n    summaries=[]\n    for keys,f in predictions.groupby([\'configuration\',\'task\',\'evaluation\',\'role\'],sort=False):summaries.append(dict(zip([\'configuration\',\'task\',\'evaluation\',\'role\'],keys))|metrics(f))\n    summary=pd.DataFrame(summaries);annual,annual_summary=annual_evaluation(predictions)\n    for name,frame in [(\'heldout_predictions.parquet\',predictions),(\'monthly_metrics.parquet\',summary),(\'annual_predictions.parquet\',annual),(\'annual_metrics.parquet\',annual_summary),(\'monthly_qualification.parquet\',qualification),(\'activity_representation.parquet\',pd.DataFrame(activity_stats))]:frame.to_parquet(output/name,index=False)\n    (output/\'split_audit.json\').write_text(json.dumps(audits,indent=2,allow_nan=False)+\'\\n\');(output/\'method.json\').write_text(json.dumps(POLICY,indent=2)+\'\\n\')\n    primary=summary[summary.configuration.eq(\'primary\')&summary.evaluation.eq(\'outer_heldout_year\')].copy()\n    report={\'research_id\':POLICY[\'research_id\'],\'input_dataset_id\':manifest[\'dataset_id\'],\'input_manifest_sha256\':digest((DATA/\'manifest.json\').read_bytes()),\n      \'primary_results\':primary.to_dict(\'records\'),\'new_results_do_not_replace_old_scores\':True,\'old_scores_status\':\'earlier dataset, not reused or certified here\',\n      \'scope\':\'regional reference-sensor observed vegetation activity; matched calendar support only\',\n      \'scientific_release_status\':\'not_enabled; coverage/overpass, uncertainty and local-allocation gates require review\',\n      \'noaa20_gap_fill_enabled\':False,\'early_history_bands_enabled\':False,\'district_custom_area_transfer_validated\':False,\n      \'coverage_verified\':False,\'production_export_enabled\':False,\'frontend_changed\':False,\'files\':{}}\n    for path in sorted(output.iterdir()):\n        if path.is_file() and path.name!=\'manifest.json\':raw=path.read_bytes();report[\'files\'][path.name]={\'bytes\':len(raw),\'sha256\':digest(raw)}\n    (output/\'manifest.json\').write_text(json.dumps(report,indent=2,allow_nan=False)+\'\\n\')\n    if summary_path is not None:\n        Path(summary_path).parent.mkdir(parents=True,exist_ok=True)\n        Path(summary_path).write_text(json.dumps({k:v for k,v in report.items() if k!=\'files\'},indent=2,allow_nan=False)+\'\\n\')\n    print(primary[[\'task\',\'role\',\'months\',\'wape\',\'relative_bias\',\'empirical_coverage_80\',\'empirical_coverage_90\']].to_string(index=False))\n    return {\'report\':report,\'predictions\':predictions,\'summary\':summary,\'annual\':annual,\'annual_summary\':annual_summary,\'qualification\':qualification,\'audits\':audits,\'output\':output}\n\n','frozen_transfer_research.py','exec'),research_namespace)
research_namespace['DATA']=DATA
OUTPUT=WORK/'results';OUTPUT.mkdir(exist_ok=True)
results=research_namespace['run'](output=OUTPUT)


## Inspect the results
Read both outer year holdouts and the stricter chronological test. WAPE is total absolute error / total observed reference activity. Annual ranks require at least three fully eligible years; annual intervals remain unavailable. Good regional prediction scores do not authorize NOAA-20 gap filling or early-history bands.

In [ ]:
import pandas as pd
from IPython.display import display
display(results['summary'])
display(results['annual_summary'])
display(results['qualification'].query("configuration=='primary' and year>=2022").groupby(['task','year'])[['eligible_month','eligible_year']].agg({'eligible_month':'sum','eligible_year':'max'}))
print('Release:',results['report']['scientific_release_status'])


## Download one results ZIP
Includes predictions, residuals, uncertainty diagnostics, qualifications, split logs and a file-hash manifest. Keep it as a research artifact; it does not replace frontend data.

In [ ]:
from datetime import datetime,timezone
archive_path=WORK/('emberline_atlas_lk_transfer_research_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.iterdir()):
        if path.is_file():archive.write(path,path.name)
print('Saved:',archive_path.resolve())
print('SHA256:',hashlib.sha256(archive_path.read_bytes()).hexdigest())
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(archive_path))
